# Step 11 – Second event check: 3–5 Oct 2025 floods
Lists every Sentinel-1 pass (with satellite: 1A or 1C) and Sentinel-2 clear % over the same two study areas, 20 Sep – 25 Oct 2025.
CPU is fine, about 2–3 minutes. No export, nothing is saved.

In [ ]:
PROJECT = 'your-ee-project-id'   # <-- your Earth Engine Cloud project ID
import ee, pandas as pd
ee.Authenticate(); ee.Initialize(project=PROJECT)
AOIS = {'kathmandu': ee.Geometry.Rectangle([85.20, 27.58, 85.48, 27.78]),
        'koshi_terai': ee.Geometry.Rectangle([86.85, 26.45, 87.15, 26.75])}
START, END = '2025-09-20', '2025-10-25'
PEAK = pd.Timestamp('2025-10-05')

## 1. Sentinel-1 passes (`days from peak`: negative = before 5 Oct)

In [ ]:
def s1_table(aoi):
    col = (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi).filterDate(START, END)
           .filter(ee.Filter.eq('instrumentMode', 'IW'))
           .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
           .map(lambda im: im.set('cover', im.geometry().intersection(aoi, 1).area(1).divide(aoi.area(1)))))
    rows = col.reduceColumns(ee.Reducer.toList(5), ['system:time_start', 'platform_number', 'orbitProperties_pass',
                                                     'relativeOrbitNumber_start', 'cover']).get('list').getInfo()
    df = pd.DataFrame(rows, columns=['t', 'satellite', 'pass', 'orbit', 'cover'])
    df['time (UTC)'] = pd.to_datetime(df.t, unit='ms')
    df['date'] = df['time (UTC)'].dt.strftime('%Y-%m-%d')
    out = (df.groupby(['date', 'satellite', 'pass', 'orbit'], as_index=False)
             .agg(cover=('cover', 'sum'), time=('time (UTC)', 'min')))
    out['cover'] = out.cover.clip(upper=1).round(2)
    out['time (UTC)'] = out.time.dt.strftime('%H:%M'); out = out.drop(columns='time')
    out['days from peak'] = (pd.to_datetime(out.date) - PEAK).dt.days
    return out.sort_values('date').reset_index(drop=True)

for name, aoi in AOIS.items():
    print(f'\n=== {name} ==='); display(s1_table(aoi))

## 2. Sentinel-2 clear share

In [ ]:
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
def clear_table(aoi):
    col = s2.filterBounds(aoi).filterDate(START, END)
    days = sorted(set(col.aggregate_array('system:time_start').map(lambda t: ee.Date(t).format('YYYY-MM-dd')).getInfo()))
    rows = []
    for d in days:
        im = col.filterDate(ee.Date(d), ee.Date(d).advance(1, 'day')).mosaic(); scl = im.select('SCL')
        clr = scl.eq(2).Or(scl.gte(4).And(scl.lte(7))).unmask(0).reduceRegion(ee.Reducer.mean(), aoi, 60).get('SCL')
        rows.append({'date': d, 'clear %': round(ee.Number(clr).multiply(100).getInfo())})
    df = pd.DataFrame(rows); df['days from peak'] = (pd.to_datetime(df.date) - PEAK).dt.days
    return df

for name, aoi in AOIS.items():
    print(f'\n=== {name} ==='); display(clear_table(aoi))